# 04 — Baseline LSTM Training (4 variants × 3 seeds)

Trains **4 baseline LSTMs × 3 seeds = 12 runs** by invoking `src/train_LSTM_baseline.py` per variant per seed. Each variant runs Optuna once (seed 42) and reuses those hyperparameters for seeds 43 / 44 via the new `--fixed-hparams` CLI arg, saving ~60 % Colab time vs rerunning Optuna each seed.

| Variant | Features | Seeds trained | Artifact names |
|---|---|---|---|
| **O** | stationary (5) | 42, 43, 44 | `lstm_baseline_O_seed{N}.pt` |
| **A** | stationary + sentiment (7) | 42, 43, 44 | `lstm_baseline_seed{N}.pt` |
| **H** | A + `p_volatile` (8) | 42, 43, 44 | `lstm_baseline_H_seed{N}.pt` |
| **B** | A + VIX family (10) | 42, 43, 44 | `lstm_baseline_B_seed{N}.pt` |

## Why 3 seeds

Per `supplementary/discussion.md`, single-seed ensemble MSE swung +53 % between reruns with identical config (Optuna TPE non-determinism + CPU PyTorch float noise). k=3 gives a coarse-but-honest mean±std across seeds, disclosing the training variance.

Aggregation: **simple mean** of per-seed predictions in nb 06 / nb 07. Paper claims quote `mean ± std` across the 3 seeds.

## Why Optuna-once-per-variant

Optuna is expensive (~60 % of training wall time). Hyperparameters are a property of the *data + architecture + objective*, not the seed; reusing seed-42's best_params for seeds 43 / 44 doesn't bias the variance estimate because it leaves the two known sources of seed-level variance (Optuna trial order + weight init + float-noise in retrain) both active across seeds. If Optuna's TPE state itself were a material variance source, we'd need to rerun it — we're betting it isn't, which matches the standard "architecture fixed, retrain with new seed" convention in deep-learning research.

## Prerequisites (from upstream notebooks)

- `data/processed/train.parquet` etc. with the variant H feature `p_volatile` injected (nb 03 § 10c).
- Full training window: ~3,690 rows for variants O/A/H; ~2,030 rows for variant B (VIX3M history limit).


In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_baseline.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
sys.path.insert(0, str(REPO_ROOT))

import config

print("Repo root:", REPO_ROOT)
print("Script   :", SCRIPT)


Repo root: /content/repo
Script   : /content/repo/src/train_LSTM_baseline.py


## Prerequisite check

Verifies train.parquet contains every feature column required by any variant (including `p_volatile` for variant H). Raises a clear error if nb 03's cell 10c hasn't run.


In [2]:
train_df = pd.read_parquet(config.DATA_PROCESSED / "train.parquet")

required_cols = (
    set(config.LSTM_VARIANT_O_FEATURES)
    | set(config.LSTM_VARIANT_A_FEATURES)
    | set(config.LSTM_VARIANT_H_FEATURES)
    | set(config.LSTM_VARIANT_B_FEATURES)
    | {config.LSTM_TARGET}
)
missing = sorted(required_cols - set(train_df.columns))
if missing:
    raise RuntimeError(
        f"train.parquet missing columns: {missing}. "
        f"Run nb 01 (features) and nb 03 (§10c p_volatile injection) first."
    )

def _trainable_rows(df, feats):
    return int(df[list(feats) + [config.LSTM_TARGET]].dropna().shape[0])

summary = pd.DataFrame({
    "variant":    ["O", "A", "H", "B"],
    "n_features": [len(config.LSTM_VARIANT_O_FEATURES),
                   len(config.LSTM_VARIANT_A_FEATURES),
                   len(config.LSTM_VARIANT_H_FEATURES),
                   len(config.LSTM_VARIANT_B_FEATURES)],
    "trainable_rows": [_trainable_rows(train_df, config.LSTM_VARIANT_O_FEATURES),
                       _trainable_rows(train_df, config.LSTM_VARIANT_A_FEATURES),
                       _trainable_rows(train_df, config.LSTM_VARIANT_H_FEATURES),
                       _trainable_rows(train_df, config.LSTM_VARIANT_B_FEATURES)],
})
print("Per-variant trainable rows (after dropping NaN):")
print(summary.to_string(index=False))


Per-variant trainable rows (after dropping NaN):
variant  n_features  trainable_rows
      O           5            3691
      A           7            3691
      H           8            3690
      B          10            2034


## Configure variants × seeds

Edit `VARIANTS` or `SEEDS` below to skip configs for partial reruns. Default runs all 12.


In [3]:
VARIANTS = [
    {"name": "O", "features": config.LSTM_VARIANT_O_FEATURES, "prefix_base": "lstm_baseline_O"},
    {"name": "A", "features": config.LSTM_VARIANT_A_FEATURES, "prefix_base": "lstm_baseline"},
    {"name": "H", "features": config.LSTM_VARIANT_H_FEATURES, "prefix_base": "lstm_baseline_H"},
    {"name": "B", "features": config.LSTM_VARIANT_B_FEATURES, "prefix_base": "lstm_baseline_B"},
]
SEEDS = [42, 43, 44]

for v in VARIANTS:
    for s in SEEDS:
        print(f"  variant {v['name']}  seed {s}  → {v['prefix_base']}_seed{s}.pt  ({len(v['features'])} feats)")
print(f"\nTotal runs: {len(VARIANTS) * len(SEEDS)}")


  variant O  seed 42  → lstm_baseline_O_seed42.pt  (5 feats)
  variant O  seed 43  → lstm_baseline_O_seed43.pt  (5 feats)
  variant O  seed 44  → lstm_baseline_O_seed44.pt  (5 feats)
  variant A  seed 42  → lstm_baseline_seed42.pt  (7 feats)
  variant A  seed 43  → lstm_baseline_seed43.pt  (7 feats)
  variant A  seed 44  → lstm_baseline_seed44.pt  (7 feats)
  variant H  seed 42  → lstm_baseline_H_seed42.pt  (8 feats)
  variant H  seed 43  → lstm_baseline_H_seed43.pt  (8 feats)
  variant H  seed 44  → lstm_baseline_H_seed44.pt  (8 feats)
  variant B  seed 42  → lstm_baseline_B_seed42.pt  (10 feats)
  variant B  seed 43  → lstm_baseline_B_seed43.pt  (10 feats)
  variant B  seed 44  → lstm_baseline_B_seed44.pt  (10 feats)

Total runs: 12


## Training sweep

Per variant: seed 42 does full Optuna and writes an hparams JSON; seeds 43 / 44 load that JSON via `--fixed-hparams` and skip Optuna.

Runs are streamed live. If one variant's seed 42 fails, the whole variant is skipped (seeds 43/44 need its JSON). Other variants continue.

Expected wall time per *variant* (all 3 seeds): ~15 min Optuna + 2 × ~5 min retrain ≈ 25 min on Colab GPU. Total for 4 variants: ~90-120 min.


In [4]:
variant_outputs = {}

MODELS_DIR = config.MODELS_DIR
MODELS_DIR.mkdir(parents=True, exist_ok=True)


def _run_one_seed(script_path, args_list, label):
    """Run one training invocation, streaming output live. Returns captured stdout."""
    cmd = [sys.executable, "-u", str(script_path), *args_list]
    print("=" * 80)
    print(f"[{label}]")
    print("Command:", " ".join(cmd))
    print("-" * 80)
    lines = []
    proc = subprocess.Popen(
        cmd, cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    rc = proc.wait()
    return rc, "".join(lines)


for v in VARIANTS:
    vname = v["name"]
    hparams_json_path = MODELS_DIR / f"hparams_{v['prefix_base']}.json"
    variant_outputs[vname] = {"seeds": {}}

    for seed_idx, seed in enumerate(SEEDS):
        prefix = f"{v['prefix_base']}_seed{seed}"
        label = f"variant {vname}  seed {seed}"

        args_list = [
            "--features", *v["features"],
            "--output-prefix", prefix,
            "--seed", str(seed),
        ]
        # Seeds 43, 44: use seed 42's JSON to skip Optuna
        if seed_idx > 0:
            if not hparams_json_path.exists():
                print(f"  [{label}]  SKIPPED — seed 42 didn't produce {hparams_json_path}")
                variant_outputs[vname]["seeds"][seed] = {"status": "skipped"}
                continue
            args_list += ["--fixed-hparams", str(hparams_json_path)]

        try:
            rc, text = _run_one_seed(SCRIPT, args_list, label)
        except Exception as exc:
            print(f"[{label}]  EXCEPTION: {exc}")
            variant_outputs[vname]["seeds"][seed] = {"status": "exception", "error": str(exc)}
            continue

        if rc != 0:
            print(f"[{label}]  FAILED (rc={rc})")
            variant_outputs[vname]["seeds"][seed] = {"status": "failed", "return_code": rc, "output": text}
            # If seed 42 failed, skip subsequent seeds for this variant.
            if seed_idx == 0:
                print(f"  Variant {vname}: seed 42 failed — skipping seeds {SEEDS[1:]}")
                break
            continue

        variant_outputs[vname]["seeds"][seed] = {"status": "ok", "output": text}

        # After seed 42 succeeds, extract its JSON and persist for seeds 43/44.
        if seed_idx == 0:
            marker = "=== Baseline LSTM results ==="
            idx = text.find(marker)
            if idx != -1:
                try:
                    blob = text[idx + len(marker):].strip()
                    results = json.loads(blob)
                    hparams_json_path.write_text(json.dumps(results, indent=2))
                    print(f"  [seed 42]  hparams JSON saved → {hparams_json_path.name}")
                except json.JSONDecodeError as exc:
                    print(f"  [seed 42]  JSON parse failed: {exc}; seeds 43/44 will skip.")


[variant O  seed 42]
Command: /usr/bin/python3 -u /content/repo/src/train_LSTM_baseline.py --features log_return abs_return oc_return intraday_range relative_volume_21d --output-prefix lstm_baseline_O_seed42 --seed 42
--------------------------------------------------------------------------------


22:19:23 | INFO    | train_LSTM_baseline | Features: ['log_return', 'abs_return', 'oc_return', 'intraday_range', 'relative_volume_21d']
22:19:23 | INFO    | train_LSTM_baseline | Target  : realized_vol_21d
22:19:23 | INFO    | train_LSTM_baseline | Split sizes — train=3710, val=1089, test=1481
22:19:23 | INFO    | train_LSTM_baseline | n_features=5 | rows — train=3691 val=1089 test=1481
22:19:23 | INFO    | train_LSTM_baseline | Starting Optuna study with 20 trials…


22:19:28 | INFO    | train_LSTM_baseline | epoch   1 | train_log_mse=2.976401 | val_mse_raw=0.00004664 | best=0.00004664


22:19:29 | INFO    | train_LSTM_baseline | epoch   2 | train_log_mse=0.151301 | val_mse_raw=0.00004741 | best=0.00004664


22:19:29 | INFO    | train_LSTM_baseline | epoch   3 | train_log_mse=0.126267 | val_mse_raw=0.00004552 | best=0.00004552


...

22:56:26 | INFO    | train_LSTM_baseline | epoch  29 | train_log_mse=0.039173 | val_mse_raw=0.00004837 | best=0.00003722


22:56:27 | INFO    | train_LSTM_baseline | epoch  30 | train_log_mse=0.041733 | val_mse_raw=0.00005446 | best=0.00003722


22:56:28 | INFO    | train_LSTM_baseline | epoch  31 | train_log_mse=0.036082 | val_mse_raw=0.00004736 | best=0.00003722


22:56:29 | INFO    | train_LSTM_baseline | epoch  32 | train_log_mse=0.034059 | val_mse_raw=0.00005050 | best=0.00003722


22:56:29 | INFO    | train_LSTM_baseline | epoch  33 | train_log_mse=0.029965 | val_mse_raw=0.00005267 | best=0.00003722
22:56:29 | INFO    | train_LSTM_baseline | Early stopping at epoch 33
22:56:29 | INFO    | train_LSTM_baseline | Final retrain — best val MSE (raw scale): 0.00003722
22:56:29 | INFO    | train_LSTM_baseline | Test metrics: {'MSE': 1.9314065866637975e-05, 'RMSE': 0.0043947771191596985, 'MAE': 0.0028576727490872145, 'n_test_windows': 1461}
22:56:29 | INFO    | train_LSTM_baseline | Saved model → /content/repo/models/lstm_baseline_B_seed44.pt
22:56:29 | INFO    | train_LSTM_baseline | Saved scaler → /content/repo/models/lstm_baseline_B_seed44_scaler.joblib

=== Baseline LSTM results ===
{
  "best_params": {
    "hidden_size": 64,
    "n_layers": 3,
    "dropout": 0.24485885415905723,
    "lr": 0.009818247569037455,
    "batch_size": 32,
    "seq_len": 21
  },
  "best_val_mse_raw": NaN,
  "retrained_val_mse_raw": 3.721631583175622e-05,
  "test_metrics": {
    "MSE": 1.93

## Aggregate per-variant results: mean ± std across 3 seeds


In [5]:
import numpy as np

marker = "=== Baseline LSTM results ==="
per_seed_results = {}

for vname, vdata in variant_outputs.items():
    per_seed_results[vname] = {}
    for seed, info in vdata["seeds"].items():
        if info.get("status") != "ok":
            continue
        text = info["output"]
        idx = text.find(marker)
        if idx == -1: continue
        try:
            per_seed_results[vname][seed] = json.loads(text[idx + len(marker):].strip())
        except json.JSONDecodeError:
            continue

# Build a long-form table: one row per (variant, seed)
rows = []
for vname, seeds_dict in per_seed_results.items():
    for seed, r in seeds_dict.items():
        rows.append({
            "variant":  vname,
            "seed":     seed,
            "test_MSE": r["test_metrics"]["MSE"],
            "test_RMSE": r["test_metrics"]["RMSE"],
            "test_MAE": r["test_metrics"]["MAE"],
            "seq_len":  r["best_params"]["seq_len"],
            "hidden_size": r["best_params"]["hidden_size"],
        })
per_seed_df = pd.DataFrame(rows)
print("Per-(variant, seed) test metrics:")
display(per_seed_df)

# Aggregate: mean ± std per variant across seeds
agg_rows = []
for vname, seeds_dict in per_seed_results.items():
    if not seeds_dict:
        continue
    mses  = [r["test_metrics"]["MSE"]  for r in seeds_dict.values()]
    rmses = [r["test_metrics"]["RMSE"] for r in seeds_dict.values()]
    maes  = [r["test_metrics"]["MAE"]  for r in seeds_dict.values()]
    agg_rows.append({
        "variant":     vname,
        "n_seeds":     len(seeds_dict),
        "MSE_mean":    np.mean(mses),
        "MSE_std":     np.std(mses, ddof=1) if len(mses) > 1 else 0.0,
        "RMSE_mean":   np.mean(rmses),
        "RMSE_std":    np.std(rmses, ddof=1) if len(rmses) > 1 else 0.0,
        "MAE_mean":    np.mean(maes),
        "MAE_std":     np.std(maes, ddof=1) if len(maes) > 1 else 0.0,
    })
agg_df = pd.DataFrame(agg_rows).set_index("variant")
print("\nPer-variant aggregate (mean ± std across seeds):")
agg_df


Per-(variant, seed) test metrics:


,variant,seed,test_MSE,test_RMSE,test_MAE,seq_len,hidden_size
0,O,42,0.000015,0.003855,0.002461,21,64
1,O,43,0.000021,0.004564,0.003077,21,64
2,O,44,0.000017,0.004180,0.002664,21,64
3,A,42,0.000013,0.003655,0.002471,42,128
4,A,43,0.000015,0.003867,0.002660,42,128
5,A,44,0.000019,0.004317,0.002814,42,128
6,H,42,0.000016,0.003981,0.002575,21,64
7,H,43,0.000015,0.003832,0.002545,21,64
8,H,44,0.000014,0.003724,0.002470,21,64
9,B,42,0.000018,0.004269,0.002796,21,64



Per-variant aggregate (mean ± std across seeds):


,n_seeds,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std
variant,,,,,,,
O,3,0.000018,2.991382e-06,0.004200,0.000355,0.002734,0.000314
A,3,0.000016,2.710291e-06,0.003946,0.000338,0.002648,0.000172
H,3,0.000015,9.942422e-07,0.003846,0.000129,0.002530,0.000054
B,3,0.000018,1.443841e-06,0.004240,0.000171,0.002775,0.000095


## Summary (fill in after execution)

From the aggregate table above:

- **MSE_std / MSE_mean** is the relative standard error across seeds. If it's > 10 %, single-seed metrics are unreliable (justifies multi-seed reporting in the paper). Historical reference: ensemble MSE swung +53 % between single-seed reruns.
- Variants with **smaller MSE_std** are more stable under seed variation — they're candidates for the paper's "robust finding" column.

DM significance testing across variants happens in nb 06 (within-variant) and nb 07 (cross-variant), operating on the mean-of-seed-predictions.


## Saved artifacts

| File | Description |
|---|---|
| `models/lstm_baseline_{O,H,B}_seed{42,43,44}.pt` | 9 variant baselines (O, H, B × 3 seeds) |
| `models/lstm_baseline_seed{42,43,44}.pt` | 3 variant-A baselines (no variant suffix by convention) |
| `models/lstm_baseline*_scaler.joblib` | Matching StandardScaler for each `.pt` |
| `models/hparams_lstm_baseline*.json` | Seed 42's best_params — used by seeds 43/44 via `--fixed-hparams` |

Total: 12 checkpoints + 12 scalers + 4 hparams JSONs.

Each `.pt` contains `state_dict`, `hyperparameters`, `features`, `target`, `seed`, and (for seeds 43/44) `fixed_hparams_source`.
